In [ ]:
import json
import re
from pathlib import Path

import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

In [ ]:
# Resolve paths from either the repository root or the model/ folder.
repo_root = Path.cwd()
if not (repo_root / "keys" / "keys.json").exists() and repo_root.name == "model":
    repo_root = repo_root.parent

keys_path = repo_root / "keys" / "keys.json"
with keys_path.open("r", encoding="utf-8") as f:
    keys = json.load(f)

login(token=keys["HF_TOKEN"])
model_id = keys["HF_LLAMA_FT_MODEL"]
output_tree_path = repo_root / "model" / "tree.xml"

In [ ]:
# Use bf16 when the active CUDA device supports it; otherwise fall back to fp16.
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
else:
    compute_dtype = torch.float16

# 4-bit quantization keeps inference lightweight for the released 1B model.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

In [ ]:
# The HF token from keys/keys.json is used above for authentication.
# No interactive Hugging Face login is required when HF_TOKEN is set.

In [ ]:
# Load the fine-tuned BTGenBot-2 model from the Hugging Face repo in HF_LLAMA_FT_MODEL.
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
)

In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

# Keep generation bounded to one behavior tree while allowing the model to sample valid alternatives.
generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
    "do_sample": True,
}

In [ ]:
system_content = """
You are a helpful assistant that can assist with creating behavior trees.
Your task:
- Convert the provided summary of a behavior into an XML-formatted behavior tree.
- Ensure the behavior tree matches the description in the summary.
- The behavior tree must be compatible with the BehaviorTree.CPP library.
- Only use the actions and parameters provided in the action list below the summary.

Output Requirements:
- Output only the XML representation of the behavior tree. Do not include explanations, comments, or any additional text.
- Ensure all actions and parameters strictly match the provided list.
- If possible limit the use of SubTrees.

Please generate the behavior tree based on the summary and action list provided.
"""

In [ ]:
user_content = """
This behavior tree manages a robot doing navigation. Initially, it moves to the location "Warehouse Left". Then, it moves to the "Warehouse Forklift".

Actions: [MoveTo (parameters: location)]
"""

messages = [
    {"role": "system", "content": system_content},
    {"role": "user", "content": user_content},
]

# The generated text should contain only XML, but extraction below guards against extra text.
output = pipe(messages, **generation_args)
generated_text = output[0]["generated_text"]
print(generated_text)

In [ ]:
print(f"Repository root: {repo_root}")
print(f"Generated tree will be saved to: {output_tree_path}")

In [ ]:
# Extract the last complete <root>...</root> block from the model response.
pattern = r"<root[^>]*>.*?</root>"
matches = re.findall(pattern, generated_text, re.DOTALL)
if not matches:
    raise ValueError("No complete <root>...</root> behavior tree found in the generated output.")

final_tree = matches[-1]
print(final_tree)

In [ ]:
# Save the generated behavior tree for simulator or BehaviorTree.CPP use.
output_tree_path.parent.mkdir(parents=True, exist_ok=True)
with output_tree_path.open("w", encoding="utf-8") as f:
    f.write(final_tree)